# Python com MongoDB
## NoSQL Database: Connection, CRUD and Queries

[◀ Anterior](34_MySQL.ipynb) | [📖 Indice](00_Index.ipynb) |

---

## 🎯 Objetivos de Aprendizagem

- Entender o que e MongoDB e quando usa-lo (NoSQL vs SQL)
- Instalar e conectar ao MongoDB com `pymongo`
- Criar bancos de dados e colecoes (collections)
- Inserir documentos (BSON/JSON)
- Consultar com `find()` e filtros
- Ordenar, limitar, atualizar e deletar documentos
- Usar queries avancadas e operadores do MongoDB

## 📝 Introducao

**MongoDB** e um banco de dados NoSQL orientado a documentos. Em vez de tabelas e linhas, ele armazena **documentos JSON** em **colecoes** (collections). E ideal para dados nao estruturados, esquemas flexiveis e aplicacoes que precisam escalar horizontalmente.

### MongoDB vs MySQL

| MySQL (Relacional) | MongoDB (NoSQL) |
|---------------------|------------------|
| Tabelas e linhas | Colecoes e documentos |
| Esquema rigido (colunas fixas) | Esquema flexivel (schemaless) |
| SQL (Structured Query Language) | Queries baseadas em JSON |
| JOINs entre tabelas | Documentos aninhados ou referencias |
| Transacoes ACID | Eventual consistency (ou ACID desde 4.0) |

```bash
pip install pymongo
```

> ⚠️ **Pre-requisito**: MongoDB instalado e rodando (localmente ou MongoDB Atlas na nuvem).

## 📚 Explicacao Detalhada

### Conexao com MongoDB

```python
from pymongo import MongoClient

cliente = MongoClient("mongodb://localhost:27017/")
db = cliente["meu_banco"]
colecao = db["minha_colecao"]
```

### Principais Operacoes

| Operacao | MongoDB | Descricao |
|----------|---------|-----------|
| Criar BD | `cliente["nome"]` | Criado automaticamente ao inserir |
| Criar Colecao | `db["nome"]` | Criada automaticamente ao inserir |
| Inserir 1 | `insert_one(doc)` | Insere um documento |
| Inserir varios | `insert_many([docs])` | Insere varios documentos |
| Buscar todos | `find()` | Retorna cursor com todos os docs |
| Buscar com filtro | `find(filtro)` | Filtra documentos |
| Buscar 1 | `find_one(filtro)` | Retorna o primeiro match |
| Ordenar | `.sort("campo", 1/-1)` | 1=ASC, -1=DESC |
| Limitar | `.limit(n)` | Limita numero de resultados |
| Atualizar 1 | `update_one(filtro, novo)` | Atualiza primeiro match |
| Atualizar varios | `update_many(filtro, novo)` | Atualiza todos os matches |
| Deletar 1 | `delete_one(filtro)` | Remove primeiro match |
| Deletar varios | `delete_many(filtro)` | Remove todos os matches |
| Deletar colecao | `colecao.drop()` | Remove a colecao inteira |

### Operadores de Query

```python
# Igualdade
{"nome": "Joao"}

# Maior que
{"idade": {"$gt": 18}}

# Em uma lista
{"curso": {"$in": ["CC", "SI"]}}

# Expressao regular
{"nome": {"$regex": "^A"}}  # nomes que comecam com A

# Combinados (AND)
{"idade": {"$gt": 18}, "nota": {"$gte": 7.0}}
```

## 💻 Exemplos

In [ ]:
# Exemplo 1: Conectando e criando banco/colecao
from pymongo import MongoClient

try:
    cliente = MongoClient("mongodb://localhost:27017/")
    # Listar bancos existentes
    print("Bancos disponiveis:")
    for db_name in cliente.list_database_names():
        print(f"  - {db_name}")

    # Criar/acessar banco e colecao
    db = cliente["escola_mongo"]
    colecao = db["alunos"]
    print("\nBanco 'escola_mongo' e colecao 'alunos' prontos!")

    cliente.close()
except Exception as err:
    print(f"Erro: {err}")
    print("(Certifique-se de que o MongoDB esta rodando em localhost:27017)")

In [ ]:
# Exemplo 2: INSERT — Inserindo documentos
try:
    cliente = MongoClient("mongodb://localhost:27017/")
    db = cliente["escola_mongo"]
    colecao = db["alunos"]

    # Inserir um unico documento
    aluno1 = {
        "nome": "Maria Silva",
        "idade": 20,
        "curso": "Ciencia da Computacao",
        "notas": [8.5, 9.0, 7.5],
        "ativo": True
    }
    resultado = colecao.insert_one(aluno1)
    print(f"Documento inserido. ID: {resultado.inserted_id}")

    # Inserir varios documentos
    varios_alunos = [
        {"nome": "Joao Santos", "idade": 22, "curso": "Sistemas de Informacao", "notas": [7.0, 6.5, 8.0], "ativo": True},
        {"nome": "Ana Pereira", "idade": 19, "curso": "Engenharia de Software", "notas": [9.5, 9.8, 9.0], "ativo": True},
        {"nome": "Pedro Costa", "idade": 25, "curso": "Ciencia da Computacao", "notas": [6.0, 5.5, 7.0], "ativo": False},
        {"nome": "Lucia Oliveira", "idade": 21, "curso": "Sistemas de Informacao", "notas": [8.0, 9.0, 8.5], "ativo": True},
    ]
    resultado = colecao.insert_many(varios_alunos)
    print(f"{len(resultado.inserted_ids)} documentos inseridos.")

    cliente.close()
except Exception as err:
    print(f"Erro: {err}")

In [ ]:
# Exemplo 3: FIND — Consultando documentos
try:
    cliente = MongoClient("mongodb://localhost:27017/")
    db = cliente["escola_mongo"]
    colecao = db["alunos"]

    # find() — todos os documentos
    print("Todos os alunos:")
    for aluno in colecao.find():
        print(f"  {aluno['nome']} - {aluno['curso']} - Notas: {aluno['notas']}")

    # find_one() — primeiro documento
    print(f"\nPrimeiro aluno: {colecao.find_one()['nome']}")

    cliente.close()
except Exception as err:
    print(f"Erro: {err}")

In [ ]:
# Exemplo 4: FIND com filtros (Query)
try:
    cliente = MongoClient("mongodb://localhost:27017/")
    db = cliente["escola_mongo"]
    colecao = db["alunos"]

    # Filtro por igualdade
    print("1. Alunos de Ciencia da Computacao:")
    for aluno in colecao.find({"curso": "Ciencia da Computacao"}):
        print(f"   {aluno['nome']}")

    # Operador $gt (maior que)
    print("\n2. Alunos com idade > 20:")
    for aluno in colecao.find({"idade": {"$gt": 20}}):
        print(f"   {aluno['nome']} - {aluno['idade']} anos")

    # Operador $in (esta na lista)
    print("\n3. Alunos em CC ou ES:")
    for aluno in colecao.find({"curso": {"$in": ["Ciencia da Computacao", "Engenharia de Software"]}}):
        print(f"   {aluno['nome']} - {aluno['curso']}")

    # Regex
    print("\n4. Alunos com nome comecando com 'M':")
    for aluno in colecao.find({"nome": {"$regex": "^M"}}):
        print(f"   {aluno['nome']}")

    cliente.close()
except Exception as err:
    print(f"Erro: {err}")

In [ ]:
# Exemplo 5: SORT e LIMIT
try:
    cliente = MongoClient("mongodb://localhost:27017/")
    db = cliente["escola_mongo"]
    colecao = db["alunos"]

    # Ordenar por nome (1 = ascendente)
    print("Ordenado por nome (ASC):")
    for aluno in colecao.find().sort("nome", 1):
        print(f"  {aluno['nome']}")

    # Ordenar por idade decrescente e limitar a 3
    print("\nTop 3 mais velhos:")
    for aluno in colecao.find().sort("idade", -1).limit(3):
        print(f"  {aluno['nome']} - {aluno['idade']} anos")

    cliente.close()
except Exception as err:
    print(f"Erro: {err}")

In [ ]:
# Exemplo 6: UPDATE — Atualizando documentos
try:
    cliente = MongoClient("mongodb://localhost:27017/")
    db = cliente["escola_mongo"]
    colecao = db["alunos"]

    # update_one — atualizar primeiro match
    filtro = {"nome": "Maria Silva"}
    novos_valores = {"$set": {"notas": [9.0, 9.5, 9.0]}}
    resultado = colecao.update_one(filtro, novos_valores)
    print(f"update_one: {resultado.modified_count} documento(s) modificado(s)")

    # update_many — atualizar todos os matches
    resultado = colecao.update_many(
        {"ativo": True},
        {"$set": {"status": "regular"}}
    )
    print(f"update_many: {resultado.modified_count} documento(s) modificado(s)")

    # Incrementar campo (operador $inc)
    colecao.update_one({"nome": "Joao Santos"}, {"$inc": {"idade": 1}})
    joao = colecao.find_one({"nome": "Joao Santos"})
    print(f"\nJoao apos incrementar idade: {joao['idade']} anos")

    cliente.close()
except Exception as err:
    print(f"Erro: {err}")

In [ ]:
# Exemplo 7: DELETE — Removendo documentos
try:
    cliente = MongoClient("mongodb://localhost:27017/")
    db = cliente["escola_mongo"]
    colecao = db["alunos"]

    # delete_one
    resultado = colecao.delete_one({"nome": "Pedro Costa"})
    print(f"delete_one: {resultado.deleted_count} documento(s) removido(s)")

    # Contar documentos restantes
    print(f"\nDocumentos restantes: {colecao.count_documents({})}")

    # delete_many (exemplo: remover inativos)
    # resultado = colecao.delete_many({"ativo": False})
    # print(f"delete_many: {resultado.deleted_count} documento(s) removido(s)")

    cliente.close()
except Exception as err:
    print(f"Erro: {err}")

In [ ]:
# Exemplo 8: DROP collection e operacoes avancadas
try:
    cliente = MongoClient("mongodb://localhost:27017/")
    db = cliente["escola_mongo"]

    # Listar colecoes
    print("Colecoes no banco escola_mongo:")
    for col in db.list_collection_names():
        print(f"  - {col}")

    # Agregacao - Pipeline (media de idade por curso)
    colecao = db["alunos"]
    pipeline = [
        {"$group": {
            "_id": "$curso",
            "media_idade": {"$avg": "$idade"},
            "total_alunos": {"$sum": 1},
            "alunos": {"$push": "$nome"}
        }},
        {"$sort": {"media_idade": -1}}
    ]

    print("\nAgregacao - Estatisticas por curso:")
    for grupo in colecao.aggregate(pipeline):
        print(f"  {grupo['_id']}:")
        print(f"    Media de idade: {grupo['media_idade']:.1f}")
        print(f"    Total de alunos: {grupo['total_alunos']}")
        print(f"    Alunos: {', '.join(grupo['alunos'])}")

    # DROP collection (comente para preservar os dados)
    # colecao.drop()
    # print("\nColecao 'alunos' removida!")
    print("\nDROP comentado para seguranca. Descomente se quiser limpar.")

    cliente.close()
except Exception as err:
    print(f"Erro: {err}")

## ✅ Exercicios Resolvidos

In [ ]:
# Exercicio 1: Funcao para inserir e buscar com seguranca
def inserir_aluno_mongo(colecao, dados):
    resultado = colecao.insert_one(dados)
    return resultado.inserted_id

def buscar_por_curso(colecao, curso):
    return list(colecao.find({"curso": curso}))

# Exemplo (descomente para testar):
# aluno = {"nome": "Carlos Teste", "idade": 23, "curso": "Ciencia da Computacao", "notas": [8.0, 7.5]}
# novo_id = inserir_aluno_mongo(colecao, aluno)
# print(f"Aluno inserido com ID: {novo_id}")
# cc_alunos = buscar_por_curso(colecao, "Ciencia da Computacao")
# print(f"Alunos de CC: {len(cc_alunos)}")

In [ ]:
# Exercicio 2: Relatorio com aggregation pipeline
pipeline_relatorio = [
    {"$match": {"ativo": True}},
    {"$project": {
        "nome": 1,
        "curso": 1,
        "media_notas": {"$avg": "$notas"},
        "maior_nota": {"$max": "$notas"}
    }},
    {"$sort": {"media_notas": -1}},
    {"$limit": 3}
]

print("Top 3 alunos por media de notas (ativos):")
try:
    cliente = MongoClient("mongodb://localhost:27017/")
    colecao = cliente["escola_mongo"]["alunos"]
    for aluno in colecao.aggregate(pipeline_relatorio):
        print(f"  {aluno['nome']}: Media={aluno['media_notas']:.2f}, Melhor={aluno['maior_nota']}")
    cliente.close()
except Exception as err:
    print(f"Erro: {err}")

## 📋 Exercicios Propostos

### Facil

1. Conecte ao MongoDB e crie um banco `biblioteca` com colecao `livros`.
2. Insira 5 documentos de livros com campos: titulo, autor, ano, genero, preco.
3. Use `find()` para listar todos os livros.
4. Use `find_one()` para encontrar um livro especifico.

### Medio

5. Use filtros para encontrar livros com preco > R$ 50, ordenados por preco decrescente.
6. Atualize o preco de um livro usando `update_one()` e `$set`.
7. Remova livros de um genero especifico com `delete_many()`.
8. Use `sort()` e `limit()` para encontrar os 3 livros mais caros.
9. Crie uma collection `autores` e simule um "join" manual (busque livros de um autor buscando o ID em ambas as collections).

### Dificil

10. Use aggregation pipeline para calcular: total de livros por genero, preco medio por genero, autor com mais livros.
11. Implemente indices com `create_index()` e compare performance de queries com e sem indice.

## 🏆 Desafios

1. **API de Biblioteca com MongoDB**: Crie um mini sistema de biblioteca usando MongoDB. Funcionalidades: cadastrar livro (com titulo, autor, ISBN, genero, quantidade), emprestar (reduzir quantidade, registrar historico), devolver, buscar por titulo/autor/ISBN, listar emprestados. Use documentos aninhados para o historico de emprestimos.

2. **Migracao SQL para NoSQL**: Modele um sistema de e-commerce: crie o esquema no MySQL (tabelas: clientes, produtos, pedidos, itens_pedido) e depois recrie o mesmo no MongoDB com documentos aninhados (pedido contem itens como subdocumentos). Compare as vantagens e desvantagens de cada abordagem.

## 📌 Resumo

- **MongoDB**: banco NoSQL orientado a documentos JSON/BSON
- **Colecoes** = tabelas; **Documentos** = linhas (mas com esquema flexivel)
- **`pymongo`**: `MongoClient` -> `db` -> `collection` -> operacoes
- **`insert_one/insert_many`**: inserir documentos
- **`find(filtro)`**: consultar com filtros estilo JSON
- **`update_one/update_many`**: `$set`, `$inc`, `$push`
- **`delete_one/delete_many`**: remover documentos
- **`sort()`, `limit()`, `skip()`**: ordenar e paginar
- **Aggregation Pipeline**: operacoes complexas (group, project, match, sort)

## 💡 Curiosidades

- MongoDB foi criado em 2007 pela empresa **10gen** (hoje MongoDB Inc.). O nome vem de "humongous" (imenso).
- Documentos sao armazenados em formato **BSON** (Binary JSON), que e mais eficiente que JSON texto.
- MongoDB e usado por empresas como **Uber, eBay, Google, Facebook** e **Adobe**.
- Ao contrario do MySQL, MongoDB nao precisa de `CREATE DATABASE` ou `CREATE TABLE` — bancos e colecoes sao criados automaticamente ao inserir dados.
- O **Aggregation Pipeline** do MongoDB e como um cano (pipe) onde os documentos passam por varios estagios de transformacao — similar ao `|` do shell Unix.

## ✅ Boas Praticas

- Projete o esquema pensando nos **padroes de acesso**, nao em normalizacao (como no SQL)
- Prefira documentos aninhados a JOINs em NoSQL
- Use indices (`create_index`) para campos frequentemente consultados
- Sempre feche a conexao: `cliente.close()`
- Use `try/except` para tratar `pymongo.errors`
- Para producao, use `MongoClient` com connection pooling e autenticacao

## ⚠️ Erros Comuns

| Erro | Exemplo | Correcao |
|------|---------|----------|
| MongoDB nao iniciado | `ServerSelectionTimeoutError` | Iniciar mongod: `mongod --dbpath /data/db` |
| Usar `$set` em vez de sobrescrever | `update_one({}, {campo: valor})` remove outros campos | Usar `{"$set": {campo: valor}}` |
| Esquecer que `find()` retorna cursor | Iterar cursor 2x retorna vazio na segunda | Converter para lista: `list(cursor)` |
| Modelar NoSQL como SQL | Criar muitas colecoes com referencias | Aninhar documentos e desnormalizar quando necessario |
| Nao usar projecao | `find()` retorna documentos inteiros | Usar segundo parametro: `find({}, {"nome": 1})` |
| Ignorar indices | Queries lentas em colecoes grandes | Criar indices com `create_index()` |

## 📖 Referencias

- [W3Schools - MongoDB Get Started](https://www.w3schools.com/python/python_mongodb_getstarted.asp)
- [W3Schools - MongoDB Create DB](https://www.w3schools.com/python/python_mongodb_create_db.asp)
- [W3Schools - MongoDB Collection](https://www.w3schools.com/python/python_mongodb_create_collection.asp)
- [W3Schools - MongoDB Insert](https://www.w3schools.com/python/python_mongodb_insert.asp)
- [W3Schools - MongoDB Find](https://www.w3schools.com/python/python_mongodb_find.asp)
- [W3Schools - MongoDB Query](https://www.w3schools.com/python/python_mongodb_query.asp)
- [W3Schools - MongoDB Sort](https://www.w3schools.com/python/python_mongodb_sort.asp)
- [W3Schools - MongoDB Delete](https://www.w3schools.com/python/python_mongodb_delete.asp)
- [W3Schools - MongoDB Drop Collection](https://www.w3schools.com/python/python_mongodb_drop_collection.asp)
- [W3Schools - MongoDB Update](https://www.w3schools.com/python/python_mongodb_update.asp)
- [W3Schools - MongoDB Limit](https://www.w3schools.com/python/python_mongodb_limit.asp)
- [Documentacao Oficial PyMongo](https://pymongo.readthedocs.io/)
- [MongoDB Manual](https://docs.mongodb.com/manual/)

---

## 📄 Creditos

**Compilado e desenvolvido por Luciano Vilas Boas Espiridiao**

📧 luciano.espiridiao@ifmg.edu.br

Baseado no site da W3Schools.

---

[◀ Anterior](34_MySQL.ipynb) | [📖 Indice](00_Index.ipynb)